In [1]:
import torch
from tabpfn.model.loading import load_model_criterion_config
from tabpfn import TabPFNClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
### PyTorch interface

In [2]:
model_name = "TabPFN-Wide-8k" 
assert model_name in ["TabPFN-Wide-1.5k", "TabPFN-Wide-5k", "TabPFN-Wide-8k", "TabPFNv2"], f"Model name {model_name} not recognized."
checkpoint_path = f"./models/{model_name}_submission.pt"

In [3]:
model, _, _ = load_model_criterion_config(
            model_path=None,
            check_bar_distribution_criterion=False,
            cache_trainset_representation=False,
            which='classifier',
            version='v2',
            download=True,
        )
if model_name != "TabPFNv2":
    model.features_per_group = 1
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint)

---
### Sklearn interface (requires PyTorch model to be loaded)

In [4]:
from tabpfnwide.patches import fit
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
tabpfn_classifier = TabPFNClassifier(n_estimators=1, device=device, ignore_pretraining_limits=True) # Turn off ensembling 
# Patch the fit method to implement an easy way to fit TabPFN-Wide while using the sklearn interface
setattr(TabPFNClassifier, 'fit', fit)

# Example data
X, y = make_classification(n_samples=50, n_features=10, n_informative=2, n_redundant=2, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
tabpfn_classifier.fit(X_train, y_train, model=model)
tabpfn_classifier.score(X_test, y_test)

0.9

In [6]:
import numpy as np
import torch

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

from tabpfn import TabPFNClassifier
from tabpfnwide.patches import fit
from tabpfnwide.utils import PredictionResults
from tabpfn.model.loading import load_model_criterion_config

setattr(TabPFNClassifier, "fit", fit)

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model, _, _ = load_model_criterion_config(
    model_path=None,
    check_bar_distribution_criterion=False,
    cache_trainset_representation=False,
    which="classifier",
    version="v2",
    download=True,
)

model.features_per_group = 1
checkpoint = torch.load("models/TabPFN-Wide-5k_submission.pt", map_location=device)
model.load_state_dict(checkpoint)

data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target

y = LabelEncoder().fit_transform(y)

seeds = [0, 1, 2, 3, 4]
folds = 5

accs, f1s, aucs = [], [], []

for seed in seeds:
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    clf = TabPFNClassifier(device=device, n_estimators=1, ignore_pretraining_limits=True)

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        X_train, X_test = X[tr], X[te]
        y_train, y_test = y[tr], y[te]

        clf.fit(X_train, y_train, model=model)

        probs = clf.predict_proba(X_test)

        pred_res = PredictionResults(y_test, probs)

        acc = pred_res.get_classification_report(print_report=False)["accuracy"]
        f1 = pred_res.get_f1_score(average="weighted")

        auc = roc_auc_score(y_test, probs[:, 1])

        accs.append(acc)
        f1s.append(f1)
        aucs.append(auc)

        print(f"seed={seed} fold={fold} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")

print("\n===== FINAL =====")
print(f"Acc: {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"F1 : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")
print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

seed=0 fold=0 acc=0.9649 f1=0.9649 auc=0.9859
seed=0 fold=1 acc=0.9912 f1=0.9912 auc=1.0000
seed=0 fold=2 acc=0.9649 f1=0.9645 auc=0.9967
seed=0 fold=3 acc=0.9825 f1=0.9824 auc=0.9980
seed=0 fold=4 acc=0.9735 f1=0.9734 auc=0.9953
seed=1 fold=0 acc=0.9737 f1=0.9735 auc=0.9980
seed=1 fold=1 acc=0.9737 f1=0.9737 auc=0.9984
seed=1 fold=2 acc=0.9825 f1=0.9824 auc=0.9884
seed=1 fold=3 acc=0.9737 f1=0.9735 auc=0.9921
seed=1 fold=4 acc=0.9912 f1=0.9911 auc=0.9966
seed=2 fold=0 acc=0.9737 f1=0.9735 auc=0.9961
seed=2 fold=1 acc=0.9825 f1=0.9825 auc=0.9984
seed=2 fold=2 acc=0.9561 f1=0.9558 auc=0.9851
seed=2 fold=3 acc=0.9912 f1=0.9912 auc=0.9997
seed=2 fold=4 acc=0.9823 f1=0.9824 auc=0.9990
seed=3 fold=0 acc=0.9649 f1=0.9645 auc=0.9961
seed=3 fold=1 acc=0.9737 f1=0.9737 auc=0.9856
seed=3 fold=2 acc=0.9912 f1=0.9912 auc=0.9974
seed=3 fold=3 acc=0.9825 f1=0.9824 auc=0.9987
seed=3 fold=4 acc=0.9823 f1=0.9822 auc=0.9987
seed=4 fold=0 acc=0.9825 f1=0.9824 auc=0.9882
seed=4 fold=1 acc=0.9474 f1=0.9474